# 📦 P1 — Data Engineer
## Speech-to-Retrieval System (SRS) — INPT Project

**Objectif** : Construire et préparer le dataset `speech → document`

### Plan
1. Installation & Setup
2. Collecte audio (Common Voice / LibriSpeech)
3. Prétraitement audio (wav, 16kHz, mono, nettoyage)
4. Construction corpus texte (Wikipedia + docs)
5. Découpage en chunks (512 tokens)
6. Création des paires `pairs.csv`
7. Validation & statistiques du dataset
8. Export pour P2 et P3


## ✅ CELLULE 1 — Installation

In [ ]:
!pip install -q datasets
!pip install -q librosa soundfile torchaudio
!pip install -q transformers
!pip install -q wikipedia-api
!pip install -q nltk
!pip install -q pandas tqdm
!pip install -q noisereduce

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✅ Installation terminée')

## ✅ CELLULE 2 — Imports & Configuration

In [ ]:
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import torchaudio
import wikipediaapi
from pathlib import Path
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# ── Configuration ────────────────────────────────────────────
CONFIG = {
    # Audio
    'sample_rate'       : 16000,
    'min_duration_sec'  : 1.0,
    'max_duration_sec'  : 20.0,
    'target_audio_count': 5000,

    # Texte
    'chunk_size_tokens' : 512,
    'chunk_overlap'     : 50,
    'target_doc_count'  : 10000,
    'tokenizer_name'    : 'sentence-transformers/all-mpnet-base-v2',

    # Paths
    'audio_dir'         : './data/audio_raw/',
    'audio_clean_dir'   : './data/audio_clean/',
    'corpus_dir'        : './data/corpus/',
    'chunks_dir'        : './data/chunks/',
    'output_dir'        : './data/output/',
}

for p in CONFIG.values():
    if isinstance(p, str) and p.startswith('./data'):
        os.makedirs(p, exist_ok=True)

print('⚙️  Configuration:')
for k, v in CONFIG.items():
    print(f'   {k:25s} = {v}')

## ✅ CELLULE 3 — Collecte audio : Common Voice (English)

In [ ]:
# ════════════════════════════════════════════════════════════
# SOURCE 1 : Mozilla Common Voice (English)
# ════════════════════════════════════════════════════════════

print('📥 Téléchargement Common Voice (English)...')
print('   (streaming=True pour éviter de tout télécharger)\n')

cv_dataset = load_dataset(
    'mozilla-foundation/common_voice_11_0',
    'en',
    split='train',
    streaming=True,
    trust_remote_code=True
)

N_CV = 2000  # nombre d'échantillons à prendre de Common Voice

cv_samples = []
for i, sample in enumerate(tqdm(cv_dataset, total=N_CV, desc='Common Voice')):
    if i >= N_CV:
        break

    # L'audio est un dict {'array': np.array, 'sampling_rate': int}
    audio_array  = np.array(sample['audio']['array'], dtype=np.float32)
    original_sr  = sample['audio']['sampling_rate']
    sentence     = sample['sentence']

    # Resample à 16kHz si nécessaire
    if original_sr != CONFIG['sample_rate']:
        audio_array = librosa.resample(
            audio_array,
            orig_sr=original_sr,
            target_sr=CONFIG['sample_rate']
        )

    cv_samples.append({
        'id'       : f'cv_{i:05d}',
        'audio'    : audio_array,
        'text'     : sentence,
        'source'   : 'common_voice',
        'duration' : len(audio_array) / CONFIG['sample_rate'],
    })

print(f'\n✅ {len(cv_samples)} samples Common Voice collectés')

## ✅ CELLULE 4 — Collecte audio : LibriSpeech

In [ ]:
# ════════════════════════════════════════════════════════════
# SOURCE 2 : LibriSpeech (clean-100 split)
# ════════════════════════════════════════════════════════════

print('📥 Téléchargement LibriSpeech (train-clean-100)...')

libri_dataset = load_dataset(
    'librispeech_asr',
    'clean',
    split='train.100',
    streaming=True,
    trust_remote_code=True
)

N_LIBRI = 3000

libri_samples = []
for i, sample in enumerate(tqdm(libri_dataset, total=N_LIBRI, desc='LibriSpeech')):
    if i >= N_LIBRI:
        break

    audio_array = np.array(sample['audio']['array'], dtype=np.float32)
    original_sr = sample['audio']['sampling_rate']
    text        = sample['text']

    if original_sr != CONFIG['sample_rate']:
        audio_array = librosa.resample(
            audio_array,
            orig_sr=original_sr,
            target_sr=CONFIG['sample_rate']
        )

    libri_samples.append({
        'id'      : f'libri_{i:05d}',
        'audio'   : audio_array,
        'text'    : text,
        'source'  : 'librispeech',
        'duration': len(audio_array) / CONFIG['sample_rate'],
    })

# Fusionner
all_samples = cv_samples + libri_samples
print(f'\n✅ Total collecté : {len(all_samples)} samples audio')
print(f'   Common Voice : {len(cv_samples)}')
print(f'   LibriSpeech  : {len(libri_samples)}')

## ✅ CELLULE 5 — Prétraitement & Nettoyage audio

In [ ]:
# ════════════════════════════════════════════════════════════
# PRÉTRAITEMENT AUDIO
# - Filtrer trop courts / trop longs
# - Détecter bruit excessif
# - Sauvegarder en .wav 16kHz mono
# ════════════════════════════════════════════════════════════

def compute_snr(waveform: np.ndarray) -> float:
    """Estime le rapport signal/bruit (SNR) en dB."""
    signal_power = np.mean(waveform ** 2)
    # Estimation du bruit sur le percentile bas (silence)
    sorted_power = np.sort(np.abs(waveform))
    noise_power  = np.mean(sorted_power[:len(sorted_power) // 10] ** 2) + 1e-10
    snr = 10 * np.log10(signal_power / noise_power)
    return snr


def normalize_audio(waveform: np.ndarray) -> np.ndarray:
    """Normalisation peak à [-1, 1]."""
    peak = np.abs(waveform).max()
    return waveform / peak if peak > 0 else waveform


def preprocess_audio_sample(sample: dict,
                              output_dir: str,
                              min_snr_db: float = 5.0) -> dict | None:
    """
    Prétraite un sample audio et le sauvegarde en .wav.

    Returns:
        dict avec métadonnées ou None si rejeté
    """
    audio    = sample['audio']
    duration = sample['duration']

    # ── Filtres de qualité ───────────────────────────────────
    if duration < CONFIG['min_duration_sec']:
        return None  # trop court

    if duration > CONFIG['max_duration_sec']:
        # Tronquer
        max_samples = int(CONFIG['max_duration_sec'] * CONFIG['sample_rate'])
        audio = audio[:max_samples]
        duration = CONFIG['max_duration_sec']

    # ── SNR check (rejeter si trop bruité) ───────────────────
    snr = compute_snr(audio)
    if snr < min_snr_db:
        return None  # trop bruité

    # ── Normalisation ────────────────────────────────────────
    audio = normalize_audio(audio)

    # ── Sauvegarder ─────────────────────────────────────────
    filename = f"{sample['id']}.wav"
    filepath = os.path.join(output_dir, filename)
    sf.write(filepath, audio, CONFIG['sample_rate'])

    return {
        'audio_id'   : sample['id'],
        'filename'   : filename,
        'filepath'   : filepath,
        'text'       : sample['text'],
        'source'     : sample['source'],
        'duration'   : round(duration, 3),
        'snr_db'     : round(snr, 2),
    }


# ── Lancer le prétraitement ──────────────────────────────────
print('🔧 Prétraitement audio en cours...')
clean_samples = []
rejected = {'too_short': 0, 'too_noisy': 0}

for sample in tqdm(all_samples, desc='Preprocessing'):
    result = preprocess_audio_sample(sample, CONFIG['audio_clean_dir'])
    if result:
        clean_samples.append(result)
    else:
        if sample['duration'] < CONFIG['min_duration_sec']:
            rejected['too_short'] += 1
        else:
            rejected['too_noisy'] += 1

# Limiter à 5000
random.shuffle(clean_samples)
clean_samples = clean_samples[:CONFIG['target_audio_count']]

print(f'\n📊 Résultats prétraitement:')
print(f'   Total initial  : {len(all_samples)}')
print(f'   ✅ Acceptés    : {len(clean_samples)}')
print(f'   ❌ Trop courts : {rejected["too_short"]}')
print(f'   ❌ Trop bruités: {rejected["too_noisy"]}')

# Sauvegarder le manifest audio
audio_manifest = pd.DataFrame(clean_samples)
audio_manifest.to_csv(os.path.join(CONFIG['output_dir'], 'audio_manifest.csv'), index=False)
print(f'\n💾 Manifest sauvegardé : audio_manifest.csv')

## ✅ CELLULE 6 — Construction du corpus texte (Wikipedia)

In [ ]:
# ════════════════════════════════════════════════════════════
# CORPUS TEXTE — Wikipedia English
# ════════════════════════════════════════════════════════════

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='SRS-INPT-Project/1.0'
)

# Catégories thématiques pertinentes pour le projet
WIKI_TOPICS = [
    # Sciences & Tech
    'Artificial intelligence', 'Machine learning', 'Deep learning',
    'Neural network', 'Natural language processing', 'Speech recognition',
    'Computer vision', 'Reinforcement learning', 'Transformer (machine learning)',
    'Convolutional neural network', 'Recurrent neural network',
    'Generative adversarial network', 'BERT (language model)',
    'Word2vec', 'Information retrieval', 'Vector space model',
    'Embedding (machine learning)', 'Attention mechanism',
    'Transfer learning', 'Self-supervised learning',
    # Mathématiques
    'Linear algebra', 'Matrix (mathematics)', 'Eigenvalues and eigenvectors',
    'Calculus', 'Gradient descent', 'Backpropagation',
    'Probability theory', 'Bayesian statistics', 'Entropy (information theory)',
    'Fourier transform', 'Signal processing',
    # Informatique
    'Python (programming language)', 'Data structure', 'Algorithm',
    'Database', 'Cloud computing', 'Computer network',
    'Operating system', 'Software engineering', 'API',
    # Physique & Sciences
    'Quantum mechanics', 'Thermodynamics', 'Electromagnetism',
    'Relativity', 'Photon', 'Electron',
    # Économie
    'Economics', 'Supply and demand', 'Macroeconomics',
    'Microeconomics', 'Game theory', 'Behavioral economics',
    # Histoire & Culture
    'World War II', 'French Revolution', 'Renaissance',
    'Industrial Revolution', 'Cold War', 'United Nations',
    # Biologie & Médecine
    'DNA', 'Genetics', 'Evolution', 'Cell biology',
    'Neuroscience', 'Immunology', 'Vaccine', 'Cancer',
]


def fetch_wikipedia_doc(title: str) -> dict | None:
    """Récupère un article Wikipedia et retourne ses sections."""
    try:
        page = wiki.page(title)
        if not page.exists():
            return None

        # Combiner toutes les sections
        full_text = page.summary
        for section in page.sections:
            if section.text.strip():
                full_text += '\n\n' + section.text

        if len(full_text.split()) < 100:  # trop court
            return None

        return {
            'title'      : title,
            'url'        : page.fullurl,
            'text'       : full_text,
            'word_count' : len(full_text.split()),
            'source'     : 'wikipedia',
        }
    except Exception as e:
        return None


# Télécharger les articles
print(f'📥 Téléchargement de {len(WIKI_TOPICS)} articles Wikipedia...')
wiki_docs = []

for topic in tqdm(WIKI_TOPICS, desc='Wikipedia'):
    doc = fetch_wikipedia_doc(topic)
    if doc:
        wiki_docs.append(doc)

print(f'✅ {len(wiki_docs)} articles récupérés')

## ✅ CELLULE 7 — Augmentation du corpus texte

In [ ]:
# ════════════════════════════════════════════════════════════
# AUGMENTATION — Atteindre 10 000 documents
# Stratégie : dataset Wikipedia HuggingFace (plus rapide)
# ════════════════════════════════════════════════════════════

print('📥 Chargement Wikipedia dataset (HuggingFace)...')

wiki_hf = load_dataset(
    'wikipedia',
    '20220301.en',
    split='train',
    streaming=True,
    trust_remote_code=True
)

# Prendre suffisamment pour atteindre 10 000 chunks au total
N_WIKI_HF = 500
hf_docs = []

for i, article in enumerate(tqdm(wiki_hf, total=N_WIKI_HF, desc='Wikipedia HF')):
    if i >= N_WIKI_HF:
        break

    text = article['text'].strip()
    if len(text.split()) < 150:
        continue

    hf_docs.append({
        'title'     : article['title'],
        'url'       : article.get('url', ''),
        'text'      : text,
        'word_count': len(text.split()),
        'source'    : 'wikipedia_hf',
    })

# Fusionner tous les documents
all_docs = wiki_docs + hf_docs

# Dédupliquer par titre
seen_titles = set()
unique_docs = []
for doc in all_docs:
    if doc['title'] not in seen_titles:
        seen_titles.add(doc['title'])
        unique_docs.append(doc)

print(f'\n📊 Corpus texte:')
print(f'   Wikipedia (manual) : {len(wiki_docs)}')
print(f'   Wikipedia (HF)     : {len(hf_docs)}')
print(f'   Total unique       : {len(unique_docs)}')
avg_words = np.mean([d['word_count'] for d in unique_docs])
print(f'   Moy. mots/doc      : {avg_words:.0f}')

## ✅ CELLULE 8 — Chunking des documents (512 tokens)

In [ ]:
# ════════════════════════════════════════════════════════════
# CHUNKING — Découper les documents en 512 tokens
# ════════════════════════════════════════════════════════════

print(f'⏳ Chargement tokenizer : {CONFIG["tokenizer_name"]}')
tokenizer = AutoTokenizer.from_pretrained(CONFIG['tokenizer_name'])


def chunk_document(doc: dict,
                    doc_idx: int,
                    chunk_size: int = 512,
                    overlap: int = 50) -> list[dict]:
    """
    Découpe un document en chunks de 512 tokens avec overlap.

    Args:
        doc        : dict avec clé 'text'
        doc_idx    : index du document
        chunk_size : tokens par chunk
        overlap    : tokens de recouvrement entre chunks

    Returns:
        Liste de dicts représentant chaque chunk
    """
    text = doc['text']
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    chunk_idx = 0
    start = 0

    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]

        # Décoder les tokens → texte
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)

        if len(chunk_text.strip()) > 50:  # ignorer les chunks trop petits
            chunk_id = f'doc{doc_idx:05d}_chunk{chunk_idx:03d}'
            chunks.append({
                'chunk_id'    : chunk_id,
                'doc_idx'     : doc_idx,
                'chunk_idx'   : chunk_idx,
                'title'       : doc['title'],
                'source'      : doc['source'],
                'text'        : chunk_text,
                'token_count' : len(chunk_tokens),
                'char_count'  : len(chunk_text),
            })
            chunk_idx += 1

        start += (chunk_size - overlap)

        if end == len(tokens):
            break

    return chunks


# Chunker tous les documents
print('\n✂️  Chunking des documents...')
all_chunks = []

for i, doc in enumerate(tqdm(unique_docs, desc='Chunking')):
    chunks = chunk_document(doc, i, CONFIG['chunk_size_tokens'], CONFIG['chunk_overlap'])
    all_chunks.extend(chunks)

    # Sauvegarder chaque chunk
    for chunk in chunks:
        chunk_path = os.path.join(CONFIG['chunks_dir'], f"{chunk['chunk_id']}.txt")
        with open(chunk_path, 'w', encoding='utf-8') as f:
            f.write(chunk['text'])

# Créer le dataframe
chunks_df = pd.DataFrame(all_chunks)
# Garder au max 10 000 chunks
if len(chunks_df) > CONFIG['target_doc_count']:
    chunks_df = chunks_df.sample(CONFIG['target_doc_count'], random_state=42).reset_index(drop=True)

# Sauvegarder
chunks_csv = os.path.join(CONFIG['output_dir'], 'corpus_chunks.csv')
chunks_df.to_csv(chunks_csv, index=False)

print(f'\n📊 Résultats chunking:')
print(f'   Total chunks   : {len(all_chunks)}')
print(f'   Chunks gardés  : {len(chunks_df)}')
print(f'   Moy. tokens    : {chunks_df["token_count"].mean():.0f}')
print(f'   Min tokens     : {chunks_df["token_count"].min()}')
print(f'   Max tokens     : {chunks_df["token_count"].max()}')
print(f'\n💾 Sauvegardé : {chunks_csv}')

## ✅ CELLULE 9 — Création des paires audio–document (pairs.csv)

In [ ]:
# ════════════════════════════════════════════════════════════
# CRÉATION DES PAIRES audio–document
# Stratégie : matching par similarité de mots-clés
# ════════════════════════════════════════════════════════════

from collections import Counter
import re

# Stopwords simples en anglais
STOPWORDS = {
    'the','a','an','is','are','was','were','be','been','being',
    'have','has','had','do','does','did','will','would','could',
    'should','may','might','shall','can','need','dare','ought',
    'to','of','in','for','on','with','at','by','from','as',
    'into','through','during','before','after','above','below',
    'and','but','or','nor','if','while','although','because',
    'this','that','these','those','i','you','he','she','it',
    'we','they','what','which','who','whom','when','where','why',
    'how','all','both','each','few','more','most','other','some'
}


def extract_keywords(text: str, n: int = 10) -> set:
    """Extrait les n mots-clés les plus importants d'un texte."""
    words = re.findall(r'[a-z]+', text.lower())
    words = [w for w in words if w not in STOPWORDS and len(w) > 3]
    return set(w for w, _ in Counter(words).most_common(n))


def compute_keyword_overlap(text1: str, text2: str) -> float:
    """Jaccard similarity entre les mots-clés de deux textes."""
    kw1 = extract_keywords(text1)
    kw2 = extract_keywords(text2)
    if not kw1 or not kw2:
        return 0.0
    intersection = len(kw1 & kw2)
    union = len(kw1 | kw2)
    return intersection / union if union > 0 else 0.0


# Créer les paires
# Pour chaque audio (texte transcrit), trouver le chunk le plus pertinent
print('🔗 Création des paires audio–document...')
print(f'   {len(clean_samples)} audios × {len(chunks_df)} chunks')
print('   (matching par mots-clés)\n')

pairs = []
NO_MATCH_THRESHOLD = 0.0  # garder toujours au moins une paire

# Sous-échantillonner les chunks pour matching rapide
sample_chunks = chunks_df.sample(min(1000, len(chunks_df)), random_state=42)

for audio in tqdm(clean_samples, desc='Matching'):
    query_text = audio['text']

    best_score  = -1
    best_chunk  = None

    for _, chunk in sample_chunks.iterrows():
        score = compute_keyword_overlap(query_text, chunk['text'])
        if score > best_score:
            best_score = score
            best_chunk = chunk

    if best_chunk is not None:
        pairs.append({
            'audio_file'   : audio['filename'],
            'audio_path'   : audio['filepath'],
            'query_text'   : query_text,
            'chunk_id'     : best_chunk['chunk_id'],
            'doc_title'    : best_chunk['title'],
            'match_score'  : round(best_score, 4),
            'source_audio' : audio['source'],
        })

pairs_df = pd.DataFrame(pairs)
pairs_csv = os.path.join(CONFIG['output_dir'], 'pairs.csv')
pairs_df.to_csv(pairs_csv, index=False)

print(f'\n✅ {len(pairs_df)} paires créées')
print(f'   Match score moyen : {pairs_df["match_score"].mean():.4f}')
print(f'   Match score max   : {pairs_df["match_score"].max():.4f}')
print(f'\n💾 Sauvegardé : {pairs_csv}')
print('\nAperçu pairs.csv:')
pairs_df[['audio_file', 'chunk_id', 'match_score']].head(5)

## ✅ CELLULE 10 — Split train/val/test

In [ ]:
# ════════════════════════════════════════════════════════════
# SPLIT TRAIN / VAL / TEST
# 80% train | 10% val | 10% test
# ════════════════════════════════════════════════════════════

from sklearn.model_selection import train_test_split

# Premier split : 80% train, 20% temp
train_df, temp_df = train_test_split(pairs_df, test_size=0.20, random_state=42)

# Deuxième split : 50% val, 50% test du temp (soit 10% / 10%)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

# Sauvegarder
train_df.to_csv(os.path.join(CONFIG['output_dir'], 'pairs_train.csv'), index=False)
val_df.to_csv(os.path.join(CONFIG['output_dir'], 'pairs_val.csv'),   index=False)
test_df.to_csv(os.path.join(CONFIG['output_dir'], 'pairs_test.csv'), index=False)

print('📊 Dataset split:')
print(f'   Train : {len(train_df):5d} paires ({len(train_df)/len(pairs_df)*100:.0f}%)')
print(f'   Val   : {len(val_df):5d} paires ({len(val_df)/len(pairs_df)*100:.0f}%)')
print(f'   Test  : {len(test_df):5d} paires ({len(test_df)/len(pairs_df)*100:.0f}%)')
print(f'\n💾 Fichiers sauvegardés dans {CONFIG["output_dir"]}')

## ✅ CELLULE 11 — Validation & Statistiques

In [ ]:
# ════════════════════════════════════════════════════════════
# VALIDATION FINALE DU DATASET
# ════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

print('=' * 55)
print('📊 RAPPORT FINAL DATASET — P1 Data Engineer')
print('=' * 55)

print(f'\n🎙️  AUDIO')
print(f'   Total fichiers    : {len(clean_samples)}')
durations = [s['duration'] for s in clean_samples]
print(f'   Durée totale      : {sum(durations)/3600:.2f} heures')
print(f'   Durée moyenne     : {np.mean(durations):.2f} s')
print(f'   Sources           :')
source_counts = pd.Series([s['source'] for s in clean_samples]).value_counts()
for src, cnt in source_counts.items():
    print(f'      {src:20s} : {cnt}')

print(f'\n📄  CORPUS TEXTE')
print(f'   Total chunks      : {len(chunks_df)}')
print(f'   Tokens moy/chunk  : {chunks_df["token_count"].mean():.0f}')
print(f'   Sources           :')
src_chunks = chunks_df['source'].value_counts()
for src, cnt in src_chunks.items():
    print(f'      {src:20s} : {cnt}')

print(f'\n🔗  PAIRES')
print(f'   Total paires      : {len(pairs_df)}')
print(f'   Train             : {len(train_df)}')
print(f'   Val               : {len(val_df)}')
print(f'   Test              : {len(test_df)}')

print(f'\n📁  FICHIERS PRODUITS')
files = [
    ('audio_manifest.csv', 'Index de tous les audios'),
    ('corpus_chunks.csv',  'Tous les chunks texte (512 tokens)'),
    ('pairs.csv',          'Toutes les paires audio–document'),
    ('pairs_train.csv',    'Paires train (80%)'),
    ('pairs_val.csv',      'Paires val (10%)'),
    ('pairs_test.csv',     'Paires test (10%)'),
]
for fname, desc in files:
    fpath = os.path.join(CONFIG['output_dir'], fname)
    size = os.path.getsize(fpath)/1024 if os.path.exists(fpath) else 0
    print(f'   ✅ {fname:30s} {size:8.1f} KB  — {desc}')

print('\n' + '=' * 55)
print('✅ Dataset P1 prêt pour P2 et P3 !')
print('   → P2 reçoit : audio_clean/ + audio_manifest.csv')
print('   → P3 reçoit : corpus_chunks.csv + pairs_train/val/test.csv')
print('=' * 55)